# Dataclasses

Hasta ahora, cada vez que creábamos una clase escribíamos a mano el `__init__`,
guardando los datos uno a uno sobre `self`. Funciona, pero para clases que son
básicamente **contenedores de datos** se vuelve repetitivo: mucho código que solo
sirve para asignar atributos.

En este vídeo veremos las **dataclasses**: una forma de crear clases de datos
escribiendo mucho menos código, dejando que Python genere por nosotros el `__init__`,
el `__repr__` y la comparación con `==`.

Las dataclasses se activan con un **decorador** (`@dataclass`), así que antes de
empezar conviene entender bien qué es eso del `@`.

> 📚 Documentación oficial de referencia:
> **[dataclasses](https://docs.python.org/es/3/library/dataclasses.html)**.

---

## Concepto previo: ¿qué es un decorador?

Ya hemos visto el símbolo `@` en vídeos anteriores (`@classmethod`, `@staticmethod`) y
lo veremos otra vez hoy. Ha llegado el momento de explicar qué es.

Un **decorador** es, simplemente, una **función que recibe otra función (o una clase)
y devuelve una versión "mejorada" de ella**, añadiéndole comportamiento sin tocar su
código original.

Para entenderlo, partamos de una idea clave de Python: las funciones son **objetos**.
Se pueden guardar en variables, pasar como argumento y devolver desde otra función.

In [ ]:
def saludar():
    print("¡Hola!")


# Una función es un objeto: puedo guardarla en otra variable y llamarla
otra = saludar
otra()  # ¡Hola!

Sabiendo eso, podemos escribir una función que **recibe** otra función, la envuelve
en una nueva con comportamiento extra, y devuelve esa nueva función:

In [ ]:
def con_marco(func):
    def envoltura():
        print("=" * 20)
        func()                 # llamamos a la función original
        print("=" * 20)

    return envoltura            # devolvemos la versión "mejorada"


def saludar():
    print("¡Hola!")


# Aplicamos el decorador "a mano": pasamos saludar y nos devuelve una versión nueva
saludar_con_marco = con_marco(saludar)
saludar_con_marco()

Eso de `saludar = con_marco(saludar)` es justo lo que hace el símbolo `@`. Escribir
`@con_marco` encima de una función es **exactamente lo mismo**, pero más cómodo de
leer:

In [ ]:
@con_marco
def saludar():
    print("¡Hola!")


saludar()  # ya sale enmarcado: @con_marco la ha "decorado" al definirla

> 💡 `@decorador` colocado encima de un `def` (o un `class`) significa
> "`funcion = decorador(funcion)`". No es magia: es azúcar sintáctico para una idea
> que ya sabíamos hacer.

En la práctica casi nunca escribirás tus propios decoradores al principio, pero sí
**usarás** muchos que vienen ya hechos. De hecho, todos estos que ya conoces o
conocerás son decoradores:

| Decorador | Qué hace |
|-----------|----------|
| `@classmethod` | Convierte un método en método de clase (recibe `cls`) |
| `@staticmethod` | Convierte un método en método estático (sin `self` ni `cls`) |
| `@dataclass` | Genera `__init__`, `__repr__`, `__eq__`... en una clase de datos |

Justamente `@dataclass` es el protagonista de este vídeo.

---

## 1. El problema: demasiado código repetitivo

Imagina una clase sencilla para representar un punto en el plano. Escrita "a mano",
se ve así:

In [ ]:
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Punto(1, 2)
print(p)                  # <__main__.Punto object at 0x...>  -> poco útil
print(Punto(1, 2) == Punto(1, 2))  # False  -> ¿no deberían ser iguales?

Dos molestias muy habituales:

- Al imprimir el objeto vemos algo ilegible (`<...object at 0x...>`), no sus datos.
- Dos puntos con los **mismos** valores se consideran **distintos**, porque por
  defecto Python compara objetos por identidad, no por contenido.

Podríamos arreglarlo escribiendo `__repr__` y `__eq__` a mano... pero es justo el
tipo de código aburrido que `@dataclass` genera por nosotros.

---

## 2. `@dataclass`: menos código, más cosas gratis

Convertimos la clase en una *dataclass* con un **decorador** y declarando los campos
con su **tipo**. A cambio, Python genera automáticamente el `__init__`, el `__repr__`
y el `__eq__`:

In [ ]:
from dataclasses import dataclass


@dataclass
class Punto:
    x: int
    y: int


p = Punto(1, 2)
print(p)                           # Punto(x=1, y=2)   -> __repr__ generado
print(Punto(1, 2) == Punto(1, 2))  # True             -> __eq__ generado

Sin escribir `__init__`, ya podemos crear puntos con `Punto(1, 2)`. Y "gratis"
obtenemos:

| Método generado | Para qué sirve |
|-----------------|----------------|
| `__init__` | Recibe los campos y los asigna sobre `self` |
| `__repr__` | Muestra el objeto de forma legible: `Punto(x=1, y=2)` |
| `__eq__` | Compara dos objetos por sus **datos**, no por su identidad |

> 💡 Las anotaciones de tipo (`x: int`) son **obligatorias** en una dataclass: es lo
> que le indica a Python cuáles son los campos. No "fuerzan" el tipo en tiempo de
> ejecución (puedes pasar un texto), pero documentan la intención y las usan las
> herramientas de análisis.

---

## 3. Valores por defecto y la trampa de los mutables

Podemos dar **valores por defecto** a los campos igual que en una función:

In [ ]:
@dataclass
class Usuario:
    nombre: str
    edad: int = 0          # valor por defecto
    activo: bool = True


print(Usuario("Ana"))           # Usuario(nombre='Ana', edad=0, activo=True)
print(Usuario("Luis", 30))      # Usuario(nombre='Luis', edad=30, activo=True)

Pero, ¿recuerdas la trampa de los **atributos mutables** de un vídeo anterior? Aquí
vuelve a aparecer. Si intentamos poner una **lista** como valor por defecto, la
dataclass nos **avisa con un error** (¡muy útil!):

In [ ]:
try:

    @dataclass
    class Equipo:
        nombre: str
        miembros: list = []  # ⚠️ lista mutable como valor por defecto

except ValueError as e:
    print("Error:", e)

La solución correcta es `field(default_factory=list)`: en vez de compartir **una**
lista, le decimos a la dataclass que **fabrique una lista nueva** para cada
instancia. Es exactamente la misma idea que "el estado mutable va en `__init__`",
pero con la sintaxis de las dataclasses.

In [ ]:
from dataclasses import field


@dataclass
class Equipo:
    nombre: str
    miembros: list = field(default_factory=list)  # ✅ una lista nueva por instancia


a = Equipo("Leones")
b = Equipo("Tigres")
a.miembros.append("Ana")

print(a.miembros)  # ['Ana']
print(b.miembros)  # []  -> cada equipo tiene su propia lista

---

## 4. Dataclasses inmutables con `frozen=True`

Si pasamos `frozen=True` al decorador, los objetos quedan **congelados**: no se
pueden modificar después de crearlos. Es ideal para valores que representan algo que
**no debería cambiar** (un color, una coordenada, una configuración).

In [ ]:
@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int


rojo = Color(255, 0, 0)
print(rojo)

try:
    rojo.r = 0  # intentar modificar un objeto frozen
except Exception as e:
    print(type(e).__name__, "->", e)

> 💡 Una *dataclass* `frozen` es, además, **hashable**: se puede usar como clave de
> un diccionario o dentro de un `set`. La inmutabilidad y el *hashing* van de la mano
> (lo veremos con más calma cuando hablemos del *data model* de Python).

---

## 5. Resumen

En este vídeo hemos visto, primero, qué es un **decorador** (una función que recibe
otra función o clase y devuelve una versión mejorada; `@decorador` es azúcar para
`x = decorador(x)`), y luego las **dataclasses**, que se activan con uno:

1. `@dataclass` genera automáticamente `__init__`, `__repr__` y `__eq__`,
   ahorrándonos código repetitivo.
2. Admiten **valores por defecto**, y para los **mutables** se usa
   `field(default_factory=list)` (una lista nueva por instancia).
3. Con `frozen=True` el objeto es **inmutable** (y hashable).

| Herramienta | Para qué |
|-------------|----------|
| `@decorador` | Envolver una función/clase para añadirle comportamiento |
| `@dataclass` | Clases de datos sin escribir `__init__`/`__repr__`/`__eq__` |
| `field(default_factory=list)` | Valor por defecto mutable, seguro |
| `@dataclass(frozen=True)` | Objeto inmutable |